# Summary: Imagenette against CIFAR-100

A side-by-side summary of the three ResNet-18s trained from scratch so far, on **accuracy**, **calibration**, **coverage** (selective prediction), and **SDM**:

| run | images seen by the network | classes | calibration images per class | validation images | trained in |
|---|---|---|---|---|---|
| **Imagenette 128 × 128** | 128 × 128 | 10 | 200 | 500 (50 per class) | [notebook 08](08_imagenette_resnet18.ipynb), SDM in [09](09_sdm_resnet18_imagenette.ipynb) |
| **Imagenette 32 × 32** | 32 × 32 | 10 | 200 | 500 (50 per class) | notebook 08 with `RESOLUTION = 32`, SDM in 09 |
| **CIFAR-100** (original images) | 32 × 32 | 100 | 100 | 2,500 (25 per class) | [notebook 04](04_cifar100_resnet18_cal100.ipynb), SDM in [05](05_sdm_resnet18_cal100.ipynb) |

Nothing is trained here: every number is recomputed from the files those notebooks saved in `outputs/`. Every threshold is chosen on the **calibration** split and measured on the **validation** split, as in the earlier notebooks.

**Reading the comparison:** the two Imagenette rows differ **only in resolution**. Imagenette 32 × 32 and CIFAR-100 have the **same resolution** but a different task (10 distinct classes against 100 classes, several of them confusable). Each number comes from one training run. Validation sets are small (500 images for Imagenette), so the 95% intervals matter.

**Evaluation audit — October 29, 2026:** thresholds and risk curves now use the shared whole-confidence-group metrics. Saved outputs are cleared because historical tables used older definitions. Calibration-matched comparisons freeze the coverage budget on calibration only; validation coverages may differ. This source repair does not rerun or validate historical benchmark claims.


## 1. Settings and loading

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src/sdm_vision").is_dir())
sys.path.insert(0, str(REPO_ROOT / "src"))
from sdm_vision.metrics import choose_threshold, coverage_accuracy_curve, aurc, admitted as admission_mask, threshold_for_coverage, auroc
import hashlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import reexpress_sdm
import torch
import torch.nn.functional as F
from reexpress_sdm import SDMModel, load_artifact
from sklearn.metrics import roc_auc_score

OUTPUT_DIR = Path("../outputs")
SDM_EPOCHS, SDM_SEED = 20, 0  # as in notebooks 05 and 09
RUNS = {  # row name: (model name, SDM model name)
    "Imagenette 128 × 128": ("imagenette_resnet18", "sdm10"),
    "Imagenette 32 × 32": ("imagenette32px_resnet18", "sdm10"),
    "CIFAR-100": ("cifar100_resnet18_cal100", "sdm_fine100"),
}
# SDM fitted on the 56 CIFAR-100 classes the ResNet handles well (notebook 07), shown as a reference in section 5.
CIFAR_SELECTED = ("CIFAR-100, SDM on 56 classes", "cifar100_resnet18_cal100", "sdm_selected56_576a27e2")

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# Chart colors: one fixed color per run, used in every chart (categorical slots 1-3 of the reference palette).
COLORS = dict(zip(RUNS, ["#2a78d6", "#eb6834", "#1baf7a"]))
INK, INK_SECONDARY, INK_MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": AXIS, "axes.labelcolor": INK_SECONDARY,
    "axes.titlecolor": INK, "axes.titlesize": 11, "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "legend.frameon": False, "legend.fontsize": 8, "lines.linewidth": 2,
})
print(f"PyTorch {torch.__version__}, reexpress-sdm {reexpress_sdm.__version__}, device: {device}")

Each run is loaded from its checkpoint (temperature, settings, class names), its embeddings cache (logits and embeddings of the training, validation and calibration images), and its fitted SDM model. The SDM model is used only if its fingerprint matches the checkpoint, so a retrained ResNet never meets a stale SDM model.

In [ ]:
def score_sdm(artifact, embeddings):
    scores = SDMModel(artifact, device=str(device)).score(embeddings, nearest_exemplars=0)
    return {
        "prediction": np.array([score.prediction for score in scores]),
        "probability": np.array([score.sdm[score.prediction] for score in scores]),
        "centroid_alpha": np.array([score.centroid_region_alpha for score in scores]),
        "lower_alpha": np.array([score.lower_region_alpha for score in scores]),
        "is_ood": np.array([score.is_ood for score in scores]),
    }


def load_run(model_name, sdm_name):
    checkpoint_file = OUTPUT_DIR / f"{model_name}.pt"
    embeddings_file = OUTPUT_DIR / f"{model_name}_embeddings.npz"
    sdm_file = OUTPUT_DIR / f"{model_name}_{sdm_name}_epochs{SDM_EPOCHS}_seed{SDM_SEED}.sdmkitmodel"
    missing = [path.name for path in [checkpoint_file, embeddings_file, sdm_file] if not path.exists()]
    assert not missing, f"missing {missing}: run the notebooks that create them first"
    checkpoint = torch.load(checkpoint_file, map_location="cpu", weights_only=False)
    artifact = load_artifact(sdm_file)
    fingerprint = f"{model_name}_avgpool512_{hashlib.sha256(checkpoint_file.read_bytes()).hexdigest()[:16]}"
    assert artifact.manifest["representation"]["fingerprint"] == fingerprint, f"{sdm_file.name} is stale: rerun the notebook that fits it"
    with np.load(embeddings_file) as cache:
        splits = {name: {"embeddings": cache[f"{name}_embeddings"], "logits": torch.from_numpy(cache[f"{name}_logits"]),
                         "labels": cache[f"{name}_labels"]} for name in ["train", "validation", "calibration"]}
    run = {"checkpoint": checkpoint, "temperature": checkpoint["temperature"], "class_names": checkpoint["class_names"],
           "splits": splits, "artifact": artifact, "calibration_sdm": score_sdm(artifact, splits["calibration"]["embeddings"]), "sdm": score_sdm(artifact, splits["validation"]["embeddings"])}
    # Softmax predictions and confidences, raw (T = 1) and temperature-scaled, for validation and calibration.
    for name in ["validation", "calibration"]:
        logits = splits[name]["logits"]
        run[name] = {"labels": splits[name]["labels"], "logits": logits,
                     "probs_raw": logits.softmax(1).numpy(), "probs": (logits / run["temperature"]).softmax(1).numpy()}
        run[name]["prediction"] = run[name]["probs"].argmax(1)
        run[name]["confidence_raw"] = run[name]["probs_raw"].max(1)
        run[name]["confidence"] = run[name]["probs"].max(1)
        run[name]["correct"] = run[name]["prediction"] == run[name]["labels"]
    return run


runs = {name: load_run(*names) for name, names in RUNS.items()}
for name, run in runs.items():
    print(f"{name}: {len(run['splits']['train']['labels']):,} training, {len(run['calibration']['labels']):,} calibration, "
          f"{len(run['validation']['labels']):,} validation images; temperature {run['temperature']:.2f}; "
          f"SDM regions at α = {[round(region['alpha'], 2) for region in run['artifact'].manifest['regions']] or 'none'}")

### Shared tools

The same definitions as notebooks 04–09: the Wilson interval, the 15-bin expected calibration error (ECE), threshold selection on calibration (plain: the answered images are at least the target accuracy; conservative: the lower end of their 95% Wilson interval is), and the area under the risk–coverage curve (AURC).

In [ ]:
def wilson_interval(successes, n, z=1.96):
    successes, n = np.asarray(successes, dtype=float), np.asarray(n, dtype=float)
    p = successes / n
    denominator = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denominator
    half_width = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denominator
    return center - half_width, center + half_width


def with_interval(correct):
    correct = np.asarray(correct)
    if len(correct) == 0:
        return "-"
    low, high = wilson_interval(correct.sum(), len(correct))
    return f"{correct.mean():.1%} ({low:.1%} – {high:.1%})"


def calibration_bins(confidence, correct, n_bins=15):
    edges = np.linspace(0, 1, n_bins + 1)
    which = np.clip(np.digitize(confidence, edges[1:-1]), 0, n_bins - 1)
    table = pd.DataFrame({"bin": which, "confidence": confidence, "correct": correct}).groupby("bin").agg(
        count=("correct", "size"), confidence=("confidence", "mean"), accuracy=("correct", "mean"))
    ece = (table["count"] / len(confidence) * (table["confidence"] - table["accuracy"]).abs()).sum()
    return table, ece








def max_coverage_at(score, correct, target):
    # Largest share of images that can be answered, ranking by `score`, with accuracy still at or above `target`.
    coverage, accuracy = coverage_accuracy_curve(score, correct)
    meets = np.flatnonzero(accuracy >= target)
    return coverage[meets.max()] if len(meets) else 0.0

## 2. Headline numbers

One row per measure, one column per run. Sections 3–5 break each block down.

- **Chance accuracy** is 10% for Imagenette and 1% for CIFAR-100: the two tasks are not equally hard, which is the point of including Imagenette at 32 × 32.
- **Softmax coverage at 95%**: the temperature-scaled threshold chosen on calibration for 95% selective accuracy, measured on validation.
- **SDM top region**: the highest α at which SDM formed a region, with its validation coverage and accuracy. **Softmax at SDM's coverage** answers the same share of calibration images with a plain confidence threshold.

In [ ]:
def headline(run):
    val, cal, settings = run["validation"], run["calibration"], run["checkpoint"]["settings"]
    train_correct = run["splits"]["train"]["logits"].argmax(1).numpy() == run["splits"]["train"]["labels"]
    labels_t = torch.from_numpy(val["labels"]).long()
    threshold = choose_threshold(cal["confidence"], cal["correct"], 0.95)
    admit = val["confidence"] >= threshold if threshold is not None else np.zeros(len(val["labels"]), bool)
    regions = run["artifact"].manifest["regions"]
    row = {
        "resolution": f"{settings.get('crop_size', 32)} × {settings.get('crop_size', 32)}",
        "classes (chance accuracy)": f"{len(run['class_names'])} ({1 / len(run['class_names']):.0%})",
        "calibration / validation images per class": f"{settings['cal_per_class']} / {settings['val_per_class']}",
        "train accuracy": f"{train_correct.mean():.1%}",
        "validation accuracy (95% interval)": with_interval(val["correct"]),
        "temperature": f"{run['temperature']:.2f}",
        "ECE before → after temperature": f"{calibration_bins(val['confidence_raw'], val['correct'])[1]:.1%} → {calibration_bins(val['confidence'], val['correct'])[1]:.1%}",
        "NLL before → after temperature": f"{F.cross_entropy(val['logits'], labels_t).item():.3f} → {F.cross_entropy(val['logits'] / run['temperature'], labels_t).item():.3f}",
        "softmax coverage at 95% (accuracy)": f"{admit.mean():.1%} ({val['correct'][admit].mean():.1%})",
        "SDM regions at α": ", ".join(f"{region['alpha']:.2f}" for region in regions) or "none",
        "SDM top region: coverage (accuracy)": "-",
        "calibration-matched softmax: accuracy": "-",
        "AURC softmax / SDM (lower is better)": f"{aurc(val['confidence'], val['correct']):.3f} / {aurc(run['sdm']['probability'], run['sdm']['prediction'] == val['labels']):.3f}",
    }
    if regions:
        top = max(region["alpha"] for region in regions)
        sdm_admit = run["sdm"]["centroid_alpha"] >= top
        calibration_coverage = (run["calibration_sdm"]["centroid_alpha"] >= top - 1e-9).mean()
        matched = admission_mask(val["confidence"], threshold_for_coverage(cal["confidence"], calibration_coverage))
        row["SDM top region: coverage (accuracy)"] = f"α ≥ {top:.2f}: {sdm_admit.mean():.1%} ({(run['sdm']['prediction'] == val['labels'])[sdm_admit].mean():.1%})"
        row["calibration-matched softmax: accuracy"] = f"{val['correct'][matched].mean():.1%}"
    return pd.Series(row)


pd.DataFrame({name: headline(run) for name, run in runs.items()})

## 3. Accuracy

Overall accuracy on each split, with 95% intervals, and how evenly it is spread over the classes. **Train accuracy** is measured without augmentation: all three ResNets nearly memorized their training images, which matters for SDM (section 5), because SDM compares new images with exactly these training images.

In [ ]:
def accuracy_table(run):
    train_correct = run["splits"]["train"]["logits"].argmax(1).numpy() == run["splits"]["train"]["labels"]
    val = run["validation"]
    per_class = pd.Series(val["correct"]).groupby(val["labels"]).mean()
    top5 = (np.argsort(-val["probs"], axis=1)[:, :5] == val["labels"][:, None]).any(1)
    return pd.Series({
        "train accuracy": with_interval(train_correct),
        "calibration accuracy": with_interval(run["calibration"]["correct"]),
        "validation accuracy": with_interval(val["correct"]),
        "validation top-5 accuracy": f"{top5.mean():.1%}",
        "train − validation gap": f"{train_correct.mean() - val['correct'].mean():.1%}",
        "per-class validation accuracy: worst / median / best": f"{per_class.min():.0%} / {per_class.median():.0%} / {per_class.max():.0%}",
        "classes below 80% on validation": f"{(per_class < 0.8).sum()} of {len(per_class)}",
    })


pd.DataFrame({name: accuracy_table(run) for name, run in runs.items()})

**Per-class accuracy.** Each dot is one class's validation accuracy; the black bar is the median. With 10 classes, Imagenette has 10 dots per run; CIFAR-100 has 100. Imagenette validation has 50 images per class and CIFAR-100 only 25, so CIFAR's dots are noisier (one image is 4 percentage points).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
rng = np.random.default_rng(0)
for position, (name, run) in enumerate(runs.items()):
    val = run["validation"]
    per_class = pd.Series(val["correct"]).groupby(val["labels"]).mean().to_numpy()
    jitter = rng.uniform(-0.18, 0.18, len(per_class))
    ax.scatter(position + jitter, per_class, s=36, color=COLORS[name], edgecolor=SURFACE, linewidth=1.5, alpha=0.9, zorder=3)
    ax.plot([position - 0.28, position + 0.28], [np.median(per_class)] * 2, color=INK, linewidth=2, zorder=4)
    ax.annotate(f"median {np.median(per_class):.0%}", (position + 0.3, np.median(per_class)), xytext=(4, 0), textcoords="offset points",
                va="center", fontsize=8, color=INK_SECONDARY)
ax.set_xticks(range(len(runs)), labels=list(runs))
ax.set(ylabel="validation accuracy of the class", ylim=(0, 1.02), xlim=(-0.5, len(runs) - 0.1),
       title="Per-class validation accuracy (one dot per class)")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
plt.tight_layout()
plt.show()

## 4. Calibration

A model is **calibrated** when its confidence matches how often it is right. The table shows, for validation:

- **mean confidence − accuracy**: positive means overconfident;
- **ECE** (15 bins) and **NLL** before and after temperature scaling (the temperature was fitted on calibration);
- the **Brier score**: the mean squared distance between the predicted probabilities and the true class (lower is better).

In [ ]:
def calibration_table(run):
    val = run["validation"]
    labels_t = torch.from_numpy(val["labels"]).long()
    one_hot = np.eye(val["probs"].shape[1])[val["labels"]]
    row = {"temperature": f"{run['temperature']:.2f}"}
    for label, confidence, probs, logits in [("T = 1", val["confidence_raw"], val["probs_raw"], val["logits"]),
                                             ("temperature-scaled", val["confidence"], val["probs"], val["logits"] / run["temperature"])]:
        row[f"mean confidence − accuracy, {label}"] = f"{confidence.mean() - val['correct'].mean():+.1%}"
        row[f"ECE, {label}"] = f"{calibration_bins(confidence, val['correct'])[1]:.1%}"
        row[f"NLL, {label}"] = f"{F.cross_entropy(logits, labels_t).item():.3f}"
        row[f"Brier score, {label}"] = f"{((probs - one_hot) ** 2).sum(1).mean():.3f}"
    return pd.Series(row)


pd.DataFrame({name: calibration_table(run) for name, run in runs.items()})

**Reliability diagrams**, one panel per run. Each point is one confidence bin: its mean confidence (x) against its accuracy (y); the point's area grows with the number of images in the bin. Points below the diagonal are overconfident. The gray line is the raw softmax (T = 1), the colored line after temperature scaling. Bins with fewer than 5 images are left out of the plot because their accuracy is mostly noise; the ECE in the legend still counts every image.

In [ ]:
fig, axes = plt.subplots(1, len(runs), figsize=(12, 4.2), sharey=True)
for ax, (name, run) in zip(axes, runs.items()):
    val = run["validation"]
    ax.plot([0, 1], [0, 1], color=AXIS, linewidth=1, linestyle="--", zorder=1)
    for confidence, color, label in [(val["confidence_raw"], INK_MUTED, "T = 1"), (val["confidence"], COLORS[name], f"T = {run['temperature']:.2f}")]:
        table, ece = calibration_bins(confidence, val["correct"])
        table = table[table["count"] >= 5]
        ax.plot(table["confidence"], table["accuracy"], color=color, linewidth=2, zorder=2, label=f"{label}: ECE {ece:.1%}")
        ax.scatter(table["confidence"], table["accuracy"], s=8 + 400 * table["count"] / len(confidence), color=color,
                   edgecolor=SURFACE, linewidth=1.5, zorder=3)
    ax.set(title=name, xlabel="mean confidence in the bin", xlim=(0, 1), ylim=(0, 1))
    ax.grid(axis="both")
    ax.legend(loc="upper left")
    ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
    ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
axes[0].set_ylabel("accuracy in the bin")
fig.suptitle("Reliability diagrams on validation (below the diagonal = overconfident)", fontsize=12, color=INK)
plt.tight_layout()
plt.show()

## 5. Coverage: answering only when confident

### Thresholds chosen on calibration, checked on validation

For three accuracy targets, the temperature-scaled softmax threshold is chosen on the calibration split (plain and conservative, as in notebook 04) and applied unchanged to validation. **Coverage** is the share of validation images answered; **met** says whether the validation accuracy reached the target.

In [ ]:
rows = {}
for name, run in runs.items():
    val, cal = run["validation"], run["calibration"]
    for target in [0.90, 0.95, 0.99]:
        row = {}
        for rule, conservative in [("plain", False), ("conservative", True)]:
            threshold = choose_threshold(cal["confidence"], cal["correct"], target, conservative)
            admit = val["confidence"] >= threshold if threshold is not None else np.zeros(len(val["labels"]), bool)
            row[f"{rule}: coverage"] = f"{admit.mean():.1%}"
            row[f"{rule}: accuracy (95% interval)"] = with_interval(val["correct"][admit])
            row[f"{rule}: met"] = "-" if not admit.any() else ("yes" if val["correct"][admit].mean() >= target else "no")
        rows[(name, f"{target:.0%}")] = row
pd.DataFrame(rows).T.rename_axis(["run", "target"])

### Coverage against accuracy, softmax and SDM

Each line ranks the validation images by a score and answers the top share of them: the **solid** line by temperature-scaled softmax confidence, the **dashed** line by SDM's probability for its predicted class. A line further up and to the right answers more images at the same accuracy. The colors are the runs, as in every chart.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
for name, run in runs.items():
    val = run["validation"]
    for score, correct, style, label in [(val["confidence"], val["correct"], "-", "softmax"),
                                         (run["sdm"]["probability"], run["sdm"]["prediction"] == val["labels"], "--", "SDM")]:
        coverage, accuracy = coverage_accuracy_curve(score, correct)
        ax.plot(coverage, accuracy, color=COLORS[name], linestyle=style, linewidth=2, label=f"{name}, {label}")
ax.axhline(0.95, color=INK_MUTED, linewidth=1, linestyle=":", zorder=0)
ax.text(0.01, 0.953, "95% target", fontsize=8, color=INK_SECONDARY, transform=ax.get_yaxis_transform())
ax.set(xlabel="coverage (share of validation images answered)", ylabel="selective accuracy", xlim=(0, 1.01), ylim=(0.6, 1.005),
       title="Coverage against accuracy on validation (solid = softmax, dashed = SDM)")
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
ax.legend(loc="lower left", ncols=2)
plt.tight_layout()
plt.show()

**Ranking quality, in numbers.** **AUROC**: how well the score ranks right answers above wrong ones (0.5 = random, 1 = perfect). **AURC**: the average error rate over all coverage levels (lower is better). **Largest coverage at 90% / 95% / 99%** reads the curves above off the validation images themselves: it describes the ranking, but it is *not* a threshold you could deploy, because it was chosen on validation.

In [ ]:
rows = {}
for name, run in runs.items():
    val = run["validation"]
    for label, score, correct in [("softmax", val["confidence"], val["correct"]),
                                  ("SDM", run["sdm"]["probability"], run["sdm"]["prediction"] == val["labels"])]:
        rows[(name, label)] = {
            "accuracy of all answers": f"{correct.mean():.1%}",
            "AUROC": f"{auroc(score, correct):.3f}",
            "AURC": f"{aurc(score, correct):.3f}",
            **{f"largest coverage at {target:.0%}": f"{max_coverage_at(score, correct, target):.1%}" for target in [0.90, 0.95, 0.99]},
        }
pd.DataFrame(rows).T.rename_axis(["run", "score"])

## 6. SDM

SDM answers only the images inside one of its **regions**. Its regions are nested (α = 0.95 inside α = 0.85, and so on), so "α ≥ 0.85" means every image in the 0.85 region or a stricter one.

The last row is the SDM model from notebook 07, fitted on the 56 CIFAR-100 classes the ResNet handles well. It is scored on **all** 2,500 CIFAR-100 validation images, so images of the 44 excluded classes count as errors when answered, as in notebook 07.

In [ ]:
def sdm_rows(name, run, prediction, labels, min_answers=5):
    sdm, val, cal = run["sdm"], run["validation"], run["calibration"]
    correct = prediction == labels
    regions = sorted((region["alpha"] for region in run["artifact"].manifest["regions"]), reverse=True)
    base = {
        "SDM accuracy (all answers)": f"{correct.mean():.1%}",
        "agrees with the ResNet": f"{(prediction == val['prediction']).mean():.1%}",
        "flagged out-of-distribution": f"{sdm['is_ood'].mean():.1%}",
    }
    if not regions:
        return {(name, "no region"): base | {"coverage": "0.0%"}}
    rows = {}
    for alpha in regions:
        admit = sdm["centroid_alpha"] >= alpha - 1e-9
        precision = pd.Series(correct[admit]).groupby(prediction[admit])
        precision = precision.mean()[precision.size() >= min_answers]
        calibration_coverage = (run["calibration_sdm"]["centroid_alpha"] >= alpha - 1e-9).mean()
        matched = admission_mask(val["confidence"], threshold_for_coverage(cal["confidence"], calibration_coverage))
        rows[(name, f"α ≥ {alpha:.2f}")] = base | {
            "coverage": f"{admit.mean():.1%}",
            "accuracy (95% interval)": with_interval(correct[admit]),
            "worst class precision": f"{precision.min():.0%}" if len(precision) else "-",
            "calibration-matched softmax": with_interval(val["correct"][matched]),
        }
    return rows


rows = {}
for name, run in runs.items():
    rows |= sdm_rows(name, run, run["sdm"]["prediction"], run["validation"]["labels"])

selected_name, selected_model, selected_sdm = CIFAR_SELECTED
selected_run = load_run(selected_model, selected_sdm)
selected = np.array([selected_run["class_names"].index(name) for name in selected_run["artifact"].manifest["configuration"]["classNames"]])
rows |= sdm_rows(selected_name, selected_run, selected[selected_run["sdm"]["prediction"]], selected_run["validation"]["labels"])

pd.DataFrame(rows).T.rename_axis(["run", "SDM answers"])

**How much does SDM answer at each α?** The share of validation images inside a region at α or stricter, for every α on SDM's ladder. A run with no region stays at 0.

In [ ]:
alphas = np.round(np.arange(0.95, 0.5, -0.05), 2)
fig, ax = plt.subplots(figsize=(8, 4.4))
for name, run in runs.items():
    coverage = [(run["sdm"]["centroid_alpha"] >= alpha - 1e-9).mean() for alpha in alphas]
    ax.plot(alphas, coverage, color=COLORS[name], linewidth=2, marker="o", markersize=8, markeredgecolor=SURFACE, markeredgewidth=2, label=name)
    ax.annotate(f"{name}: {coverage[-1]:.0%}", (alphas[-1], coverage[-1]), xytext=(6, 0), textcoords="offset points",
                va="center", fontsize=8, color=INK_SECONDARY)
ax.set(xlabel="α (SDM's reliability level)", ylabel="share of validation images answered", ylim=(-0.02, 1), xlim=(0.97, 0.45),
       title="SDM coverage at each α (left = stricter)")
ax.set_xticks(alphas)
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 7. What the comparison shows

*(Written from our run; rerunning notebooks 04–09 with a different checkpoint will change the numbers. Each run is one training seed.)*

**Accuracy: the task matters, the resolution hardly.**

- Going from 128 × 128 to 32 × 32 on Imagenette costs 2.4 points of validation accuracy (86.6% against 84.2%), and the two 95% intervals overlap.
- Going from Imagenette to CIFAR-100 at the same 32 × 32 costs 12 points (72.0%), and the spread over classes widens: 63 of 100 CIFAR classes are below 80% on validation, against 1–2 of 10 on Imagenette.
- All three ResNets nearly memorized their training images (99.3–100%).

**Calibration: the same after temperature scaling.**

- All three raw softmaxes are overconfident by 7–10 points (ECE 8.5–10.1%).
- One temperature, fitted on calibration, brings the ECE to 3.3–3.5% in every run. The Imagenette models need a larger temperature (about 2.2 against 1.35), so they were the more overconfident ones.
- How well temperature scaling works does not depend on the dataset or the resolution.

**Coverage: easier task, more images answered.**

- At a 95% target, the temperature-scaled softmax answers **75%** of validation images (Imagenette 128), **70%** (Imagenette 32) and **50%** (CIFAR-100).
- On Imagenette, every threshold chosen on calibration reached its target on validation (90%, 95% and 99%, plain and conservative).
- On CIFAR-100, the plain thresholds landed just short (88.6% against 90%, 94.5% against 95%). Only the conservative rule at 99% met its target, which is why notebook 04 recommends the conservative threshold.

**SDM: works on the easier task, never ahead of softmax.**

- SDM forms regions up to α = 0.95 on both Imagenette runs, and its answers inside them are 97–100% correct on validation. On the 100 CIFAR classes it forms none; restricted to 56 CIFAR classes, it does form regions (notebook 07).
- Wherever SDM forms a region, a softmax threshold that answers the same share of images is at least as accurate.
- On every ranking measure (AUROC, AURC, largest coverage at 90/95/99%), softmax confidence ranks right above wrong answers better than SDM's probability. The gap is small on Imagenette and large on CIFAR-100 (largest coverage at 95%: 46% against 31%).

**Bottom line:** higher resolution by itself changed little. A task with fewer, more distinct classes and more calibration images per class improved everything at once: accuracy, coverage, and SDM's ability to form regions. Even then, SDM was not more accurate than a well-calibrated softmax threshold on these in-distribution images. The case where SDM should make a difference, rejecting images from classes the model never saw, is still untested.
